# 01b — Soil available water capacity (AWC) of the cropland of each municipality

The available water capacity of the water balance is CAD = z × DTA. The DTA (mm m⁻¹) of each
municipality is the **mean AWC (m³ m⁻³ × 1000) over the temporary-crop area** of the municipality,
not the value at the seat:

- AWC per mapping unit of the soil map (asset `projects/fcoliveira/assets/AWC_br`);
- temporary-crop mask from MapBiomas Collection 10, 2024 (soybean, class 39; other temporary crops,
  class 41);
- IBGE municipal boundaries (`dados/malha_municipios_pr.geojson`);
- layer-wise particle size from MapBiomas Soil, Collection 3 (`dados/solo_camadas_pr.csv`), for the APSIM soil profile.

Municipalities without mapped temporary crops use the mean AWC of the whole municipal territory.

In [1]:
import json
from pathlib import Path

import ee
import pandas as pd

RAIZ = Path.cwd().parent
DADOS = RAIZ / "dados"
ee.Initialize(project="mapbiomas-brazil")

awc = (ee.FeatureCollection("projects/fcoliveira/assets/AWC_br")
       .filter(ee.Filter.notNull(["AWC"]))
       .reduceToImage(["AWC"], ee.Reducer.mean())
       .multiply(1000).rename("dta"))  # mm/m
uso = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection10/"
               "mapbiomas_brazil_collection10_integration_v2").select("classification_2024")
lavoura = uso.eq(39).Or(uso.eq(41))

feats = json.load(open(DADOS / "malha_municipios_pr.geojson"))["features"]
municipios = ee.FeatureCollection([ee.Feature(ee.Geometry(f["geometry"]), {"cod_ibge": int(f["properties"]["codarea"])})
                                   for f in feats])

In [2]:
img = (awc.updateMask(lavoura).rename("dta_lavoura")
       .addBands(awc.rename("dta_total"))
       .addBands(lavoura.rename("frac_lavoura")))
red = img.reduceRegions(collection=municipios, reducer=ee.Reducer.mean(), scale=90, tileScale=4).getInfo()
solo = pd.DataFrame([f["properties"] for f in red["features"]])
solo["dta_mm_m"] = solo["dta_lavoura"].fillna(solo["dta_total"])
print(solo.describe().round(2).to_string())

         cod_ibge  dta_lavoura  dta_total  frac_lavoura  dta_mm_m
count      399.00       399.00     399.00        399.00    399.00
mean   4114436.46       112.46     114.87          0.34    112.46
std       8367.35        24.27      25.50          0.24     24.27
min    4100103.00        52.73      55.91          0.00     52.73
25%    4107281.00        98.83      97.56          0.12     98.83
50%    4114351.00       120.10     122.93          0.33    120.10
75%    4121554.50       127.63     132.43          0.51    127.63
max    4128807.00       173.24     167.06          0.87    173.24


In [3]:
mun = pd.read_csv(DADOS / "municipios_pr.csv")
if "dta_ponto_mm_m" not in mun:
    mun = mun.rename(columns={"dta_mm_m": "dta_ponto_mm_m"})
mun = mun.drop(columns=["dta_mm_m", "frac_lavoura"], errors="ignore").merge(
    solo[["cod_ibge", "dta_mm_m", "frac_lavoura"]], on="cod_ibge", how="left")
print("Correlation seat DTA × cropland DTA:", round(mun["dta_ponto_mm_m"].corr(mun["dta_mm_m"]), 2))
mun.to_csv(DADOS / "municipios_pr.csv", index=False)
mun[["dta_ponto_mm_m", "dta_mm_m", "frac_lavoura"]].describe().round(2)

Correlation seat DTA × cropland DTA: 0.95


,dta_ponto_mm_m,dta_mm_m,frac_lavoura
count,399.00,399.00,399.00
mean,112.83,112.46,0.34
std,26.26,24.27,0.24
min,54.60,52.73,0.00
25%,95.40,98.83,0.12
50%,120.80,120.10,0.33
75%,130.95,127.63,0.51
max,170.50,173.24,0.87


## Layer-wise particle size (MapBiomas Soil, Collection 3) — APSIM soil profile

Clay and sand fractions (%) in 10 layers of 10 cm (0–100 cm), mapped at 30 m by MapBiomas Soil
from profiles of the Brazilian soil data repository (SoilData). Mean over the temporary-crop area of
each municipality (or over the whole municipality, when no cropland is mapped).

In [4]:
BASE_C3 = "projects/mapbiomas-public/assets/brazil/soil/collection3/mapbiomas_brazil_collection3_soil_{}_fraction_v1"
CAMADAS = ["000_010", "010_020", "020_030", "030_040", "040_050", "050_060", "060_070", "070_080", "080_090", "090_100"]
bandas = []
for v in ("clay", "sand"):
    col = ee.ImageCollection(BASE_C3.format(v))
    for c in CAMADAS:
        bandas.append(col.filter(ee.Filter.eq("depth", c)).first().rename(f"{v}_{c}"))
gran = ee.Image.cat(bandas)
red = gran.updateMask(lavoura).reduceRegions(collection=municipios, reducer=ee.Reducer.mean(), scale=90, tileScale=8).getInfo()
red_tot = gran.reduceRegions(collection=municipios, reducer=ee.Reducer.mean(), scale=90, tileScale=8).getInfo()
g = pd.DataFrame([f["properties"] for f in red["features"]]).set_index("cod_ibge")
g = g.fillna(pd.DataFrame([f["properties"] for f in red_tot["features"]]).set_index("cod_ibge"))
camadas = []
for k, c in enumerate(CAMADAS):
    camadas.append(pd.DataFrame({"cod_ibge": g.index, "camada": k + 1, "topo_cm": 10 * k, "base_cm": 10 * (k + 1),
                                 "argila_pct": g[f"clay_{c}"].to_numpy(), "areia_pct": g[f"sand_{c}"].to_numpy()}))
camadas = pd.concat(camadas, ignore_index=True).sort_values(["cod_ibge", "camada"])
camadas.to_csv(DADOS / "solo_camadas_pr.csv", index=False)
mun = pd.read_csv(DADOS / "municipios_pr.csv").drop(columns=["argila_pct", "areia_pct", "soc_pct", "densidade_g_cm3", "ph"], errors="ignore")
sup = camadas[camadas["camada"] <= 3].groupby("cod_ibge")[["argila_pct", "areia_pct"]].mean().reset_index()
mun = mun.merge(sup, on="cod_ibge", how="left")
mun.to_csv(DADOS / "municipios_pr.csv", index=False)
print("Correlation clay (0–30 cm) × cropland DTA:", round(mun["argila_pct"].corr(mun["dta_mm_m"]), 2))
camadas.groupby("camada")[["argila_pct", "areia_pct"]].mean().round(1).T

Correlation clay (0–30 cm) × cropland DTA: 0.53


camada,1,2,3,4,5,6,7,8,9,10
argila_pct,48.9,50.8,54.0,55.5,57.4,58.8,59.7,59.7,59.6,57.6
areia_pct,28.2,27.2,25.3,24.7,23.2,22.7,22.1,21.9,21.0,22.1
